In [1]:
!pip install transformers torch sentencepiece

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
  Using cached sentencepiece-0.2.1-cp314-cp314-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.29.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpma

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 206] The filename or extension is too long: 'C:\\Users\\amanp\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python314\\site-packages\\torch-2.13.0.dist-info\\licenses\\third_party\\flash-attention\\third_party\\aiter\\3rdparty\\composable_kernel\\docs'


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
text = 'I love India.'
tokens = tokenizer.tokenize(text)

print(tokens)

['i', 'love', 'india', '.']


In [19]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids) # Fixed IDs from BERT's vocabulary

# diff models -> diff vocab -> diff token id

[1045, 2293, 2634, 1012]


In [17]:
tokenizer.convert_tokens_to_ids("love")

2293

In [15]:
generated_token = tokenizer.convert_ids_to_tokens(token_ids)
print(generated_token) 

['i', 'love', 'india', '.']


In [16]:
encoding = tokenizer(text)
print(encoding)

{'input_ids': [101, 1045, 2293, 2634, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [ ]:
# [CLS] I love India . [SEP]

# Why special tokens?
# They help BERT understand where the sequence starts and ends.


# [CLS] = Classification Token
# - Added at the beginning of every input.
# - Initially it is just a normal learnable token.
# - During Self-Attention it attends to all words.
# - After passing through all Transformer layers,
#   its final embedding becomes a representation of
#   the whole sentence.
# - Used for sentence-level tasks like:
#     • Sentiment Analysis
#     • Text Classification
#     • NSP (Next Sentence Prediction)


# [SEP] = Separator Token
# - Marks the end of a sentence.
# - Separates two input sequences.

# Example:
# [CLS] I love India . [SEP]

# [CLS] I love India . [SEP]
# It is beautiful . [SEP]

In [20]:
# input = I love India.  -> length = 4 tokens

# input = I love India very much because it is a beautiful country.    -> length = 11 tokens

# we can't put both in the same batch directly!

# [
#  [101,1045,2293,2634,1012,102],
#  [101,1045,2293,2634,2200,2172,2138,2009,2003,3376,1012,102]
# ]

# cause the rows have diff lengths -> nn work with tensors and so the len of rows must be same.

# if we use pad to make same length
# [CLS] I love India . [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
# if bert starts paying attention to pad -> it would waste computation on meaningless tokens

# so we use attention mask
# bert asks -> which are real tokens and which are just padding
# input_ids : [101, 1045, 2293, 2634, 1012, 102]   -> no [PAD] tokens
# attention_mask : [1, 1, 1, 1, 1, 1]  -> mean real token, 0 -> padding token (ignore them)
# Attention Mask tells the Transformer which tokens are real (1) and which are padding (0) so it ignores the padding tokens during attention.



# [CLS]I love India. [SEP] It is beautiful.[SEP]
# sep tells bert sent 1 end here, but not explicitly tells now sent 2 starts.
# now bert -> for every token, they attach a sent ID
# 0 → Sentence A
# 1 → Sentence B

# Sentence:
# [CLS]  I  love India . [SEP]   It  is  beautiful . [SEP]

# Token Type IDs: 
#  0     0    0    0   0    0    1   1    1      1    1
# token_type_ids tell BERT which sentence (or segment) each token belongs to.